# Pregunta 3: monitoreo de fauna con YOLO (CCT-20)

Este notebook rehace todo el proyecto desde cero. Ejecutamos las celdas **de arriba abajo y en orden**: el entrenamiento se lanza en segundo plano y la celda siguiente espera a que termine.

Ajustes de la sesión que usamos: acelerador **GPU T4 x2**, **Internet: On** y **Persistence: Files only**.

## 1. Comprobaciones de la sesión

In [ ]:
# Comprobamos que esta sesión tenga GPU, memoria suficiente e Internet
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("CUDA disponible:", torch.cuda.is_available())
!free -h | head -n 2
!df -h /kaggle/working /tmp | cat

# Probamos que LILA responda (solo pedimos los encabezados)
!wget --spider -S "https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/eccv_18_annotations.tar.gz" 2>&1 | grep -E "HTTP/|Content-Length"

## 2. Escribimos los scripts

Cada celda solo crea un archivo en `/kaggle/working`.

In [ ]:
%%writefile /kaggle/working/prep_data.py
# Script de preparación: descargamos lo que falte y convertimos CCT-20 a formato YOLO
import os, json, csv, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from PIL import Image

G    = "https://storage.googleapis.com/public-datasets-lila/caltechcameratraps"
TMP  = '/tmp/cct'
RAW  = f'{TMP}/raw/eccv_18_all_images_sm'
ANN  = f'{TMP}/eccv_18_annotation_files'
OUT  = '/tmp/yolo'
KEEP = '/kaggle/working'
SPLITS = ['train', 'cis_val', 'cis_test', 'trans_val', 'trans_test']

def sh(cmd): subprocess.run(cmd, shell=True, check=True)
os.makedirs(TMP, exist_ok=True)

# 1) Si no tenemos las anotaciones o las imágenes en /tmp, las descargamos y verificamos su md5
if not os.path.isdir(ANN):
    sh(f'wget -nv -O {TMP}/ann.tar.gz {G}/eccv_18_annotations.tar.gz && md5sum {TMP}/ann.tar.gz && tar -xzf {TMP}/ann.tar.gz -C {TMP}')
    print("md5 esperado (anotaciones): 66a1f481b44aa1edadf75c9cfbd27aba")
if not os.path.isdir(RAW):
    sh(f'wget -nv -O {TMP}/img.tar.gz {G}/eccv_18_all_images_sm.tar.gz && md5sum {TMP}/img.tar.gz')
    print("md5 esperado (imágenes): 8143c17aa2a12872b66f284ff211531f")
    sh(f'mkdir -p {TMP}/raw && tar -xzf {TMP}/img.tar.gz -C {TMP}/raw && rm {TMP}/img.tar.gz')

# 2) Cargamos las cinco particiones y definimos las clases (sin 'empty', que es fondo)
data = {s: json.load(open(f'{ANN}/{s}_annotations.json')) for s in SPLITS}
CLASSES = sorted({c['name'] for d in data.values() for c in d['categories']} - {'empty'})
cid = {n: i for i, n in enumerate(CLASSES)}
print(len(CLASSES), "clases:", CLASSES)

# 3) Comprobamos que las imágenes redimensionadas conservan la proporción del original
for im in data['train']['images'][:3]:
    w, h = Image.open(f"{RAW}/{os.path.basename(im['file_name'])}").size
    print(f"real {w}x{h} (ratio {w/h:.3f}) | json {im['width']}x{im['height']} (ratio {im['width']/im['height']:.3f})")

# 4) Convertimos cada partición: etiquetas normalizadas y copia de imágenes
meta = []
def convert(split):
    d = data[split]; cname = {c['id']: c['name'] for c in d['categories']}
    boxes = {}
    for a in d['annotations']:
        if 'bbox' in a and cname[a['category_id']] != 'empty':
            boxes.setdefault(a['image_id'], []).append(a)
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)
    nb = 0
    for im in d['images']:
        W, H = im['width'], im['height']; lines = []
        for a in boxes.get(im['id'], []):
            x, y, w, h = a['bbox']
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x2 - x1 < 2 or y2 - y1 < 2: continue
            lines.append(f"{cid[cname[a['category_id']]]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        fn = os.path.basename(im['file_name']); stem = os.path.splitext(fn)[0]
        open(f'{OUT}/labels/{split}/{stem}.txt', 'w').write("\n".join(lines))
        nb += len(lines)
        meta.append([split, fn, im['location'], im.get('date_captured'), im.get('seq_id'), im.get('frame_num'), W, H, len(lines)])
    def cp(im):
        fn = os.path.basename(im['file_name']); dst = f'{OUT}/images/{split}/{fn}'
        if not os.path.exists(dst): shutil.copyfile(f'{RAW}/{fn}', dst)
    with ThreadPoolExecutor(16) as ex:
        list(tqdm(ex.map(cp, d['images']), total=len(d['images']), desc=split))
    print(f"{split}: {len(d['images'])} imágenes | {nb} cajas escritas")

for s in SPLITS: convert(s)

# 5) Guardamos lo liviano en /kaggle/working: metadatos, data.yaml y etiquetas
with open(f'{KEEP}/meta_images.csv', 'w', newline='') as f:
    wr = csv.writer(f)
    wr.writerow(['split','file','location','date_captured','seq_id','frame_num','W','H','n_boxes'])
    wr.writerows(meta)
yaml = f"path: {OUT}\ntrain: images/train\nval: images/cis_val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES))
open(f'{KEEP}/data.yaml', 'w').write(yaml)
print("listo")

In [ ]:
%%writefile /kaggle/working/train.py
# Entrenamos YOLOv8s sobre CCT-20 con protecciones contra la falta de memoria
import os, threading, time

# Limitamos los hilos y la fragmentación de memoria antes de cargar PyTorch (los procesos hijos lo heredan)
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['MALLOC_ARENA_MAX'] = '2'

def vigilar_memoria(ruta='/kaggle/working/mem.log', cada=30):
    # Cada 30 s anotamos cuánta memoria usa el contenedor, para saber qué pasó si el problema se repite
    def leer(p):
        try:
            return int(open(p).read().strip())
        except Exception:
            return -1
    while True:
        usada = leer('/sys/fs/cgroup/memory.current') / 2**30
        limite = leer('/sys/fs/cgroup/memory.max') / 2**30
        with open(ruta, 'a') as f:
            f.write(f"{time.strftime('%H:%M:%S')} memoria usada {usada:.1f} GiB de {limite:.1f} GiB\n")
        time.sleep(cada)

threading.Thread(target=vigilar_memoria, daemon=True).start()

from ultralytics import YOLO

PROJECT, NAME = '/kaggle/working/runs', 'yolov8s_cct'
last = f'{PROJECT}/{NAME}/weights/last.pt'

def entrenar_desde_cero():
    # Partimos de yolov8s preentrenado en COCO y fijamos cada decisión de entrenamiento
    model = YOLO('yolov8s.pt')
    model.train(
        data='/kaggle/working/data.yaml', epochs=20, imgsz=640, batch=16, device=0,
        workers=2, seed=0,
        mosaic=1.0, close_mosaic=5, fliplr=0.5,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1,
        plots=True, project=PROJECT, name=NAME, exist_ok=True,
    )

# Hasta 3 intentos: si el proceso cae, retomamos desde el último checkpoint
for intento in range(1, 4):
    try:
        if os.path.exists(last):
            print(f"Intento {intento}: retomando desde {last}")
            YOLO(last).train(resume=True)
        else:
            print(f"Intento {intento}: entrenamiento desde cero")
            entrenar_desde_cero()
        break
    except Exception as e:
        print(f"Intento {intento} falló: {e!r}")
        time.sleep(10)

In [ ]:
%%writefile /kaggle/working/verificar_datos.py
# Verificamos automáticamente que los datos preparados coinciden con lo que comprobamos a mano la primera vez
import os, sys, csv, hashlib

OK = True
def check(nombre, cond, detalle=''):
    # Imprimimos OK o FALLO por cada comprobación y recordamos si alguna falló
    global OK
    print(('OK     ' if cond else 'FALLO  ') + nombre + (f'  [{detalle}]' if detalle else ''))
    if not cond:
        OK = False

OUT, RAW = '/tmp/yolo', '/tmp/cct/raw/eccv_18_all_images_sm'
# Valores verificados a mano en la primera preparación: (imágenes, cajas) por partición
ESPERADO = {'train': (13553, 12617), 'cis_val': (3484, 1763), 'cis_test': (15827, 13264),
            'trans_val': (1725, 1673), 'trans_test': (23275, 18786)}

# 1) La GPU está disponible
import torch
check('GPU disponible', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

# 2) Las anotaciones descargadas son las publicadas por LILA
md5 = hashlib.md5(open('/tmp/cct/ann.tar.gz', 'rb').read()).hexdigest()
check('md5 de las anotaciones', md5 == '66a1f481b44aa1edadf75c9cfbd27aba', md5)

# 3) Están las 57,864 imágenes extraídas (el md5 de las imágenes no se puede repetir: prep_data.py borra el .tar.gz)
n_raw = len(os.listdir(RAW))
check('57,864 imágenes extraídas', n_raw == 57864, str(n_raw))

# 4) Cada partición: imágenes, etiquetas emparejadas, número de cajas, clases y coordenadas válidas
for sp, (n_img, n_box) in ESPERADO.items():
    imgs = {os.path.splitext(f)[0] for f in os.listdir(f'{OUT}/images/{sp}')}
    labs = {os.path.splitext(f)[0] for f in os.listdir(f'{OUT}/labels/{sp}')}
    check(f'{sp}: número de imágenes', len(imgs) == n_img, f'{len(imgs)} de {n_img}')
    check(f'{sp}: cada imagen tiene su etiqueta', imgs == labs, f'{len(imgs ^ labs)} sin pareja')
    cajas = mala_clase = mala_coord = 0
    for s in labs:
        for linea in open(f'{OUT}/labels/{sp}/{s}.txt').read().split('\n'):
            if not linea.strip():
                continue
            p = linea.split()
            cajas += 1
            mala_clase += not (0 <= int(p[0]) <= 14)
            mala_coord += any(not (0.0 <= float(v) <= 1.0) for v in p[1:5])
    check(f'{sp}: número de cajas', cajas == n_box, f'{cajas} de {n_box}')
    check(f'{sp}: clases 0-14 y coordenadas 0-1', mala_clase == 0 and mala_coord == 0, f'{mala_clase} clases, {mala_coord} coordenadas')

# 5) El data.yaml tiene las 15 clases y valida sobre cis_val
y = open('/kaggle/working/data.yaml').read()
n_cls = sum(1 for l in y.split('\n') if l.startswith('  ') and ':' in l)
check('data.yaml: 15 clases', n_cls == 15, str(n_cls))
check('data.yaml: val = cis_val', 'val: images/cis_val' in y)

# 6) Los metadatos están completos y las ubicaciones de train no se cruzan con las de trans
filas = list(csv.DictReader(open('/kaggle/working/meta_images.csv')))
check('meta_images.csv: 57,864 filas', len(filas) == 57864, str(len(filas)))
loc = {}
for r in filas:
    loc.setdefault(r['split'], set()).add(r['location'])
check('train no comparte ubicaciones con trans_val', not (loc['train'] & loc['trans_val']))
check('train no comparte ubicaciones con trans_test', not (loc['train'] & loc['trans_test']))

# Veredicto final; el código de salida 1 detiene el entrenamiento si algo falló
print('\nVERIFICACIÓN:', 'TODO CORRECTO' if OK else 'HAY FALLOS, NO SE ENTRENA')
sys.exit(0 if OK else 1)

In [ ]:
%%writefile /kaggle/working/eval.py
# Evaluamos best.pt en las cuatro particiones y calculamos la brecha entre cámaras vistas y nuevas
import os
os.environ['YOLO_VERBOSE'] = 'False'
import pandas as pd
from ultralytics import YOLO

W = '/kaggle/working/runs/yolov8s_cct/weights/best.pt'
SALIDA = '/kaggle/working/eval'
SPLITS = ['cis_val', 'cis_test', 'trans_val', 'trans_test']
CLASES = ['badger','bird','bobcat','car','cat','coyote','deer','dog','fox','opossum','rabbit','raccoon','rodent','skunk','squirrel']
MIN = 30   # con menos cajas que esto, el AP de una especie no es interpretable
os.makedirs(SALIDA, exist_ok=True)

def metricas_clase(box, k):
    # Leemos P, R, AP50 y AP50-95 de la clase k; si esta versión de Ultralytics no expone los vectores, usamos class_result
    try:
        return box.p[k], box.r[k], box.ap50[k], box.ap[k]
    except Exception:
        return box.class_result(k)

def contar_cajas(split):
    # Contamos las cajas reales de cada especie leyendo las etiquetas de la partición
    c = dict.fromkeys(CLASES, 0)
    d = f'/tmp/yolo/labels/{split}'
    for f in os.listdir(d):
        for l in open(f'{d}/{f}').read().split('\n'):
            if l.strip():
                c[CLASES[int(l.split()[0])]] += 1
    return c

model = YOLO(W)
glob_, filas = [], []
for sp in SPLITS:
    # Creamos un yaml propio para que la partición a evaluar sea la de validación
    yaml = f"path: /tmp/yolo\ntrain: images/train\nval: images/{sp}\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASES))
    open(f'{SALIDA}/data_{sp}.yaml', 'w').write(yaml)
    r = model.val(data=f'{SALIDA}/data_{sp}.yaml', imgsz=640, conf=0.001, iou=0.7, device=0,
                  workers=2, batch=16, plots=False, verbose=False, project=SALIDA, name=f'val_{sp}', exist_ok=True)
    glob_.append(dict(split=sp, P=r.box.mp, R=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map))
    cajas = contar_cajas(sp)
    for k, c in enumerate(r.box.ap_class_index):
        p, rec, ap50, ap = metricas_clase(r.box, k)
        filas.append(dict(split=sp, clase=CLASES[int(c)], cajas=cajas[CLASES[int(c)]], P=p, R=rec, AP50=ap50, AP50_95=ap))

G, C = pd.DataFrame(glob_), pd.DataFrame(filas)
G.to_csv(f'{SALIDA}/globales.csv', index=False); C.to_csv(f'{SALIDA}/por_clase.csv', index=False)

# Tabla por especie: cámaras vistas (cis_test) frente a cámaras nuevas (trans_test)
a = C[C.split == 'cis_test'].set_index('clase'); b = C[C.split == 'trans_test'].set_index('clase')
t = pd.DataFrame({'cajas_cis': a.cajas, 'AP50_cis': a.AP50, 'cajas_trans': b.cajas, 'AP50_trans': b.AP50,
                  'AP50_95_cis': a.AP50_95, 'AP50_95_trans': b.AP50_95})
t['caida_AP50'] = t.AP50_cis - t.AP50_trans
t['interpretable'] = (t.cajas_cis >= MIN) & (t.cajas_trans >= MIN)
t.to_csv(f'{SALIDA}/brecha_por_clase.csv')

# Promedio solo sobre las especies interpretables, para comparar manzanas con manzanas
ok = t[t.interpretable]
m = lambda sp, col: C[(C.split == sp) & C.clase.isin(ok.index)][col].mean()

print('\n===== RESUMEN =====')
print('Métricas globales (todas las especies presentes en cada partición):')
print(G.round(3).to_string(index=False))
print(f'\nBrecha sobre las {len(ok)} especies con >= {MIN} cajas en cis_test y trans_test:')
for col in ['AP50', 'AP50_95']:
    vis, nue = m('cis_test', col), m('trans_test', col)
    print(f'  {col}: cámaras vistas {vis:.3f} | nuevas {nue:.3f} | caída {vis-nue:.3f} ({100*(vis-nue)/vis:.1f} %)')
print('\nPor especie, ordenadas por caída de AP50:')
print(t.sort_values('caida_AP50', ascending=False)[['cajas_cis', 'AP50_cis', 'cajas_trans', 'AP50_trans', 'caida_AP50', 'interpretable']].round(3).to_string())

# Comprobación automática: best.pt debe reproducir, en cis_val, el mAP50 que el entrenamiento registró para su mejor época
# (Ultralytics elige best.pt por fitness = 0.1 * mAP50 + 0.9 * mAP50-95)
res = pd.read_csv('/kaggle/working/runs/yolov8s_cct/results.csv')
res.columns = res.columns.str.strip()
fit = 0.1 * res['metrics/mAP50(B)'] + 0.9 * res['metrics/mAP50-95(B)']
esperado = float(res.loc[fit.idxmax(), 'metrics/mAP50(B)'])
v = float(G[G.split == 'cis_val'].mAP50.iloc[0])
print(f'\n{"OK    " if abs(v - esperado) < 0.01 else "REVISAR"} cis_val mAP50 = {v:.3f} (el entrenamiento registró {esperado:.3f} para el mejor modelo)')
print(f'{"OK    " if len(G) == 4 else "FALLO "}evaluadas {len(G)} de 4 particiones')

In [ ]:
%%writefile /kaggle/working/monitor.py
# Módulo de monitoreo ecológico: agregamos las detecciones por fotograma en un reporte por cámara, especie y mes
import argparse, os
import pandas as pd

CLASES = ['badger','bird','bobcat','car','cat','coyote','deer','dog','fox',
          'opossum','rabbit','raccoon','rodent','skunk','squirrel']
NO_FAUNA = {'car'}   # los vehículos disparan la cámara, pero no son fauna

def preds_desde_etiquetas(meta, split, etiquetas='/tmp/yolo/labels'):
    # SOLO PARA DEPURAR: tratamos las etiquetas reales como si fueran predicciones perfectas
    filas = []
    for f in meta[meta.split == split].file:
        ruta = f"{etiquetas}/{split}/{os.path.splitext(f)[0]}.txt"
        for l in open(ruta).read().split('\n'):
            if l.strip():
                filas.append((f, CLASES[int(l.split()[0])], 1.0))
    return pd.DataFrame(filas, columns=['file', 'cls', 'conf'])

def reporte(preds, meta, camara, conf_min=0.25):
    # Preparamos los fotogramas de la cámara: mes y secuencia de cada uno
    m = meta[meta.location == camara].copy()
    m['mes'] = m.date_captured.dt.to_period('M').astype(str)
    m['seq_id'] = m.seq_id.fillna(m.file)

    # Nos quedamos con las detecciones de esa cámara que superan el umbral de confianza
    p = preds[(preds.conf >= conf_min) & preds.file.isin(m.file)].merge(m[['file', 'seq_id', 'mes']], on='file')
    fauna = p[~p.cls.isin(NO_FAUNA)]

    # Resumen de la cámara: fotos con fauna frente a vacías, y especies distintas observadas
    n_img, n_fauna = len(m), fauna.file.nunique()
    resumen = pd.Series({
        'camara': camara, 'imagenes': n_img,
        'imagenes_con_fauna': n_fauna,
        'imagenes_sin_fauna': n_img - n_fauna,
        'tasa_vacias': round(1 - n_fauna / n_img, 3),
        'imagenes_con_vehiculo': p[p.cls.isin(NO_FAUNA)].file.nunique(),
        'especies_distintas': fauna.cls.nunique(),
        'detecciones_fauna': len(fauna),
        'rango': f"{m.date_captured.min():%Y-%m-%d} a {m.date_captured.max():%Y-%m-%d}"})

    # Eventos: contamos una vez por secuencia y especie, con el máximo de individuos en un solo fotograma
    cnt = fauna.groupby(['seq_id', 'mes', 'file', 'cls']).size().rename('n').reset_index()
    ev = cnt.groupby(['seq_id', 'cls']).agg(mes=('mes', 'first'), max_individuos=('n', 'max')).reset_index()

    # Tabla por especie y tabla por especie y mes
    por_especie = (fauna.groupby('cls').size().rename('detecciones').to_frame()
                   .join(ev.groupby('cls').agg(eventos=('seq_id', 'count'), max_individuos=('max_individuos', 'max')))
                   .sort_values('eventos', ascending=False))
    por_mes = ev.pivot_table(index='cls', columns='mes', values='seq_id', aggfunc='count', fill_value=0)
    # Esfuerzo de muestreo: sin él, un mes con pocas fotos parece un mes sin actividad
    esfuerzo = pd.concat([m.groupby('mes').size().rename('imagenes'),
                          ev.groupby('mes').size().rename('eventos')], axis=1).fillna(0)
    esfuerzo['eventos_por_100_imagenes'] = (100 * esfuerzo.eventos / esfuerzo.imagenes).round(1)
    return resumen, por_especie, por_mes, esfuerzo

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--meta', default='/kaggle/working/meta_images.csv')
    ap.add_argument('--camara', type=int, required=True)
    ap.add_argument('--split', default='trans_test')
    ap.add_argument('--preds', default=None, help='CSV con columnas file,cls,conf')
    ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--salida', default='/kaggle/working/monitoreo')
    ap.add_argument('--depurar_con_etiquetas', action='store_true')
    a = ap.parse_args()

    meta = pd.read_csv(a.meta, parse_dates=['date_captured'])
    preds = preds_desde_etiquetas(meta, a.split) if a.depurar_con_etiquetas else pd.read_csv(a.preds)

    resumen, por_especie, por_mes, esfuerzo = reporte(preds, meta, a.camara, a.conf)
    os.makedirs(a.salida, exist_ok=True)
    resumen.to_csv(f'{a.salida}/resumen_cam{a.camara}.csv')
    por_especie.to_csv(f'{a.salida}/por_especie_cam{a.camara}.csv')
    por_mes.to_csv(f'{a.salida}/por_mes_cam{a.camara}.csv')
    esfuerzo.to_csv(f'{a.salida}/esfuerzo_cam{a.camara}.csv')
    print(resumen.to_string(), '\n'); print(por_especie.to_string(), '\n'); print(por_mes.to_string(), '\n'); print(esfuerzo.T.to_string())

In [ ]:
%%writefile /kaggle/working/predict.py
# Generamos predicciones sobre las imágenes de una cámara y las guardamos como CSV (file, cls, conf)
# Procesamos por lotes pequeños y liberamos memoria en cada uno, porque la versión anterior terminó "Killed" (falta de memoria)
import argparse, os, gc
import pandas as pd
import torch
from ultralytics import YOLO

# Definimos los argumentos que recibe el script
ap = argparse.ArgumentParser()
ap.add_argument('--weights', required=True)
ap.add_argument('--meta',    default='/kaggle/working/meta_images.csv')
ap.add_argument('--split',   default='trans_test')
ap.add_argument('--camara',  type=int, required=True)
ap.add_argument('--device',  default='0')
ap.add_argument('--conf',    type=float, default=0.05)   # usamos un umbral bajo; el umbral final lo aplicamos en monitor.py
ap.add_argument('--imgsz',   type=int, default=640)
ap.add_argument('--lote',    type=int, default=100)      # cuántas imágenes procesamos por lote
ap.add_argument('--salida',  required=True)
a = ap.parse_args()

def memoria_mib():
    # Leemos la memoria que usa este proceso, desde /proc
    for linea in open('/proc/self/status'):
        if linea.startswith('VmRSS'):
            return int(linea.split()[1]) / 1024
    return -1

# Elegimos las imágenes de esa cámara dentro de la partición indicada
meta = pd.read_csv(a.meta)
archivos = meta[(meta.split == a.split) & (meta.location == a.camara)].file.tolist()
rutas = [f'/tmp/yolo/images/{a.split}/{f}' for f in archivos]
print(len(rutas), "imágenes a procesar")

# Cargamos el modelo y predecimos por lotes; al terminar cada lote liberamos memoria
model = YOLO(a.weights)
filas = []
for i in range(0, len(rutas), a.lote):
    res = model.predict(rutas[i:i + a.lote], conf=a.conf, imgsz=a.imgsz, device=a.device, verbose=False)
    for f, r in zip(archivos[i:i + a.lote], res):
        for c, s in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            filas.append((f, model.names[int(c)], round(s, 4)))
    del res
    gc.collect()
    torch.cuda.empty_cache()
    # Mostramos el avance y la memoria cada diez lotes, para ver si crece
    if (i // a.lote) % 10 == 0:
        print(f"{min(i + a.lote, len(rutas))}/{len(rutas)} imágenes | memoria del proceso: {memoria_mib():.0f} MiB", flush=True)

# Guardamos todas las detecciones en un solo CSV
os.makedirs(os.path.dirname(a.salida), exist_ok=True)
pd.DataFrame(filas, columns=['file', 'cls', 'conf']).to_csv(a.salida, index=False)
print("guardado:", a.salida, "|", len(filas), "detecciones")

In [ ]:
# Comprobamos que los seis scripts quedaron escritos
import os
esperados = ['prep_data.py', 'train.py', 'verificar_datos.py', 'eval.py', 'monitor.py', 'predict.py']
for f in esperados:
    ok = os.path.exists(f'/kaggle/working/{f}') and os.path.getsize(f'/kaggle/working/{f}') > 500
    print('OK    ' if ok else 'FALLO ', f)

## 3. Preparamos y verificamos los datos

Descargamos las imágenes y las anotaciones de LILA, comprobamos sus md5, convertimos a formato YOLO y verificamos el resultado.

In [ ]:
# Instalamos la versión de Ultralytics que usamos y dejamos los datos listos: preparamos y verificamos (si algo falla, nos detenemos aquí)
!pip -q install ultralytics==8.4.192
!python /kaggle/working/prep_data.py && python /kaggle/working/verificar_datos.py && echo "DATOS LISTOS PARA ENTRENAR"

## 4. Entrenamos

Lanzamos el entrenamiento en segundo plano, esperamos a que termine y comprobamos el resultado (unos 65 minutos).

In [ ]:
# Lanzamos el entrenamiento en segundo plano (si ya hay uno corriendo, no lanzamos otro)
import subprocess

# Comprobamos de forma fiable si ya hay un entrenamiento corriendo (el corchete evita el falso positivo)
ya = subprocess.run("ps aux | grep '[t]rain.py'", shell=True, capture_output=True, text=True).stdout.strip()
if ya:
    print("Ya hay un entrenamiento corriendo:\n", ya)
else:
    # Lanzamos train.py en segundo plano y redirigimos su salida a un log
    log = open('/kaggle/working/train.log', 'w')
    proc = subprocess.Popen(
        ['python', '/kaggle/working/train.py'],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True
    )
    print("entrenamiento lanzado, PID:", proc.pid)

In [ ]:
# Esperamos a que termine el entrenamiento y mostramos el avance de las épocas (esta celda queda ocupada hasta que train.py termine)
import subprocess, time

def hay_entrenamiento():
    # Miramos si train.py sigue en ejecución (el corchete evita que grep se cuente a sí mismo)
    return bool(subprocess.run("ps aux | grep '[t]rain.py'", shell=True, capture_output=True, text=True).stdout.strip())

ultima = ''
while hay_entrenamiento():
    # Tomamos la última línea de progreso del registro (el progreso usa retornos de carro, por eso los convertimos en saltos de línea)
    progreso = subprocess.run("tr '\\r' '\\n' < /kaggle/working/train.log | grep -E '^\\s+[0-9]+/20' | tail -n 1 | cut -c1-150",
                              shell=True, capture_output=True, text=True).stdout.strip()
    if progreso and progreso != ultima:
        print(time.strftime('%H:%M:%S'), progreso, flush=True)
        ultima = progreso
    time.sleep(60)
print('El entrenamiento terminó a las', time.strftime('%H:%M:%S'))

In [ ]:
# Comprobamos si el entrenamiento terminó y cuánta memoria llegó a usar el contenedor
import os
R = '/kaggle/working/runs/yolov8s_cct'
res = f'{R}/results.csv'
epocas = len(open(res).read().strip().split('\n')) - 1 if os.path.exists(res) else 0
terminado = os.path.exists(f'{R}/weights/best.pt') and os.path.exists(f'{R}/weights/last.pt') and epocas >= 20
print(f"RESULTADO: {'ENTRENAMIENTO COMPLETO' if terminado else 'ENTRENAMIENTO INCOMPLETO'} | épocas registradas: {epocas} de 20")
if os.path.exists('/kaggle/working/mem.log'):
    picos = [float(l.split()[3]) for l in open('/kaggle/working/mem.log') if 'memoria usada' in l]
    if picos:
        print(f"Pico de memoria del contenedor: {max(picos):.1f} GiB (incluye la caché de archivos; el límite es de unos 30 GiB)")

## 5. Evaluamos

Medimos el desempeño en cámaras vistas (`cis_*`) y nuevas (`trans_*`).

In [ ]:
# Evaluamos best.pt en las cuatro particiones y mostramos solo el resumen (si falla, el final del registro)
!python /kaggle/working/eval.py > /kaggle/working/eval.log 2>&1 ; grep -q "===== RESUMEN" /kaggle/working/eval.log && sed -n '/===== RESUMEN/,$p' /kaggle/working/eval.log || tail -n 30 /kaggle/working/eval.log

## 6. Análisis de los datos

Elegimos la cámara de la demostración y comprobamos el esfuerzo de muestreo.

In [ ]:
# Cargamos los metadatos de las imágenes y miramos las cámaras de trans_test
import pandas as pd

# Cargamos los metadatos que guardó prep_data.py
meta = pd.read_csv('/kaggle/working/meta_images.csv', parse_dates=['date_captured'])
print(meta.shape)
print(meta.groupby('split').size())

# Miramos las cámaras de trans_test, que usaremos para la demostración del monitoreo
t = meta[meta.split == 'trans_test']
resumen = t.groupby('location').agg(
    imagenes=('file', 'size'),
    con_fauna=('n_boxes', lambda s: (s > 0).sum()),
    desde=('date_captured', 'min'),
    hasta=('date_captured', 'max'),
)
print(resumen)

In [ ]:
# Contamos las cajas de cada especie en cada cámara de trans_test y comprobamos los totales
import os
import pandas as pd

# Los nombres de clase, en el mismo orden que usamos en data.yaml
CLASES = ['badger','bird','bobcat','car','cat','coyote','deer','dog','fox',
          'opossum','rabbit','raccoon','rodent','skunk','squirrel']

# Leemos las etiquetas de trans_test y las unimos con la cámara de cada imagen
t = meta[meta.split == 'trans_test']
filas = []
for f, loc in zip(t.file, t.location):
    ruta = f"/tmp/yolo/labels/trans_test/{os.path.splitext(f)[0]}.txt"
    for linea in open(ruta).read().split('\n'):
        if linea.strip():
            filas.append((loc, CLASES[int(linea.split()[0])]))

gt = pd.DataFrame(filas, columns=['location', 'clase'])

# Tabla: cuántas cajas de cada especie hay en cada cámara
tabla = pd.crosstab(gt.location, gt.clase)
print(tabla.to_string())

# Cuántas especies distintas (con al menos 20 cajas) tiene cada cámara
print("\nespecies con >=20 cajas por cámara:")
print((tabla >= 20).sum(axis=1))

# Comprobamos la fila de la cámara 46 y el total contra los valores verificados antes
esperado46 = {'badger': 3, 'bird': 57, 'bobcat': 812, 'car': 0, 'cat': 17, 'coyote': 739, 'dog': 26, 'fox': 0,
              'opossum': 1105, 'rabbit': 496, 'raccoon': 985, 'rodent': 15, 'skunk': 84, 'squirrel': 10}
fila = tabla.loc[46]
print('OK    ' if all(int(fila.get(c, 0)) == v for c, v in esperado46.items()) else 'FALLO ', 'cajas por especie en la cámara 46')
print('OK    ' if int(tabla.values.sum()) == 18786 else 'FALLO ', 'cajas totales en trans_test (18786)')

In [ ]:
# Contamos cuántas imágenes tiene la cámara 46 en cada mes, para detectar huecos en los datos
m46 = meta[meta.location == 46].copy()
m46['mes'] = m46.date_captured.dt.to_period('M').astype(str)
print(m46.groupby('mes').agg(imagenes=('file', 'size'), con_fauna=('n_boxes', lambda s: (s > 0).sum())).T.to_string())

# Comprobamos contra los valores verificados antes: 6,276 imágenes, 4,242 con fauna y octubre de 2011 con solo 25 imágenes
tab = m46.groupby('mes').agg(imagenes=('file', 'size'), con_fauna=('n_boxes', lambda s: (s > 0).sum()))
print('OK    ' if tab.imagenes.sum() == 6276 and tab.con_fauna.sum() == 4242 else 'FALLO ', 'totales de la cámara 46 (6276 imágenes, 4242 con fauna)')
print('OK    ' if tab.imagenes.get('2011-10', 0) == 25 else 'FALLO ', 'octubre de 2011 con solo 25 imágenes')

## 7. Monitoreo ecológico

Probamos el módulo con las etiquetas reales como predicciones perfectas, y después con las predicciones del modelo.

In [ ]:
# Probamos el módulo con las etiquetas reales como si fueran predicciones perfectas
!python /kaggle/working/monitor.py --camara 46 --depurar_con_etiquetas --salida /kaggle/working/monitoreo_debug

# Comprobamos que el monitoreo con etiquetas reales da los valores que verificamos antes
import pandas as pd
r = pd.read_csv('/kaggle/working/monitoreo_debug/resumen_cam46.csv', index_col=0).iloc[:, 0]
esperado = {'imagenes': 6276, 'imagenes_con_fauna': 4242, 'imagenes_sin_fauna': 2034, 'especies_distintas': 12, 'detecciones_fauna': 4349}
for k, v in esperado.items():
    print('OK    ' if int(float(r[k])) == v else 'FALLO ', k, r[k])
print('OK    ' if abs(float(r['tasa_vacias']) - 0.324) < 0.001 else 'FALLO ', 'tasa_vacias', r['tasa_vacias'])

In [ ]:
# Miramos la memoria libre antes de empezar
!free -h | head -n 2

# Generamos las predicciones del modelo final sobre la cámara 46 y las pasamos por el módulo de monitoreo
!python /kaggle/working/predict.py --weights /kaggle/working/runs/yolov8s_cct/weights/best.pt --camara 46 --device 0 --salida /kaggle/working/preds/cam46.csv && python /kaggle/working/monitor.py --camara 46 --preds /kaggle/working/preds/cam46.csv --conf 0.25 --salida /kaggle/working/monitoreo

# Comprobamos que las predicciones se guardaron y que el reporte cubre las 6,276 imágenes de la cámara 46
import os
import pandas as pd
try:
    p = pd.read_csv('/kaggle/working/preds/cam46.csv')
    r = pd.read_csv('/kaggle/working/monitoreo/resumen_cam46.csv', index_col=0).iloc[:, 0]
    print('OK    ' if len(p) > 0 and {'file', 'cls', 'conf'} <= set(p.columns) else 'FALLO ', f'predicciones guardadas ({len(p)} detecciones)')
    print('OK    ' if int(float(r['imagenes'])) == 6276 else 'FALLO ', 'el reporte cubre las 6,276 imágenes de la cámara 46')
    print('OK    ' if os.path.exists('/kaggle/working/monitoreo/esfuerzo_cam46.csv') else 'FALLO ', 'tabla de esfuerzo guardada')
except FileNotFoundError as e:
    print('FALLO ', 'no se generó el archivo:', e.filename)

# Leemos el contador de procesos terminados por falta de memoria (solo lectura; si oom_kill es mayor que 0, fue la memoria)
!grep -E "oom" /sys/fs/cgroup/memory.events 2>/dev/null

In [ ]:
# Comparamos el reporte de monitoreo hecho con las etiquetas reales (la verdad) y el hecho con las predicciones del modelo
import os, subprocess
import pandas as pd

# Si todavía no tenemos el reporte con etiquetas reales en esta sesión, lo generamos (tarda unos segundos y no usa la GPU)
real_csv = '/kaggle/working/monitoreo_debug/por_especie_cam46.csv'
if not os.path.exists(real_csv):
    subprocess.run(['python', '/kaggle/working/monitor.py', '--camara', '46', '--depurar_con_etiquetas',
                    '--salida', '/kaggle/working/monitoreo_debug'], check=True, capture_output=True)

# Leemos las dos tablas por especie y las ponemos lado a lado
real = pd.read_csv(real_csv, index_col=0)
modelo = pd.read_csv('/kaggle/working/monitoreo/por_especie_cam46.csv', index_col=0)
comp = pd.DataFrame({'eventos_reales': real.eventos, 'eventos_modelo': modelo.eventos}).fillna(0).astype(int)
comp['diferencia'] = comp.eventos_modelo - comp.eventos_reales
comp['modelo_vs_real'] = (comp.eventos_modelo / comp.eventos_reales.replace(0, float('nan'))).round(2)
comp = comp.sort_values('eventos_reales', ascending=False)
comp.to_csv('/kaggle/working/monitoreo/comparacion_cam46.csv')
print(comp.to_string())
print(f"\neventos totales -> reales: {comp.eventos_reales.sum()} | modelo: {comp.eventos_modelo.sum()}")

# Comparamos también los indicadores generales de la cámara
r1 = pd.read_csv('/kaggle/working/monitoreo_debug/resumen_cam46.csv', index_col=0).iloc[:, 0]
r2 = pd.read_csv('/kaggle/working/monitoreo/resumen_cam46.csv', index_col=0).iloc[:, 0]
print()
for k in ['imagenes_con_fauna', 'tasa_vacias', 'imagenes_con_vehiculo', 'especies_distintas', 'detecciones_fauna']:
    print(f'{k:24s} reales: {r1[k]:>8s}   modelo: {r2[k]:>8s}')

# Comprobamos que la comparación quedó guardada y cubre todas las especies de las dos tablas
print('\n' + ('OK    ' if os.path.exists('/kaggle/working/monitoreo/comparacion_cam46.csv') and len(comp) == len(set(real.index) | set(modelo.index)) else 'FALLO '),
      'comparación guardada en monitoreo/comparacion_cam46.csv')

## 8. Robustez por tamaño de caja y por oclusión

Medimos cuánto cae el recall del detector según el tamaño de la caja y un proxy de oclusión, en cámaras vistas y nuevas.

In [ ]:
%%writefile /kaggle/working/robustez.py
# Analizamos la robustez del detector según el tamaño de la caja y un proxy de oclusión, en cámaras vistas (cis_test) y nuevas (trans_test)
import argparse, os, random
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# Definimos los argumentos que recibe el script
ap = argparse.ArgumentParser()
ap.add_argument('--weights', required=True)
ap.add_argument('--n',       type=int,   default=6000)   # cuántas imágenes con animal muestreamos por partición
ap.add_argument('--conf',    type=float, default=0.25)   # umbral de confianza para contar una detección (el mismo del monitoreo)
ap.add_argument('--iou',     type=float, default=0.5)    # IoU mínimo para dar por acertada una detección
ap.add_argument('--lote',    type=int,   default=100)    # cuántas imágenes procesamos por lote
ap.add_argument('--device',  default='0')
ap.add_argument('--semilla', type=int,   default=0)
ap.add_argument('--salida',  default='/kaggle/working/robustez')
a = ap.parse_args()
os.makedirs(a.salida, exist_ok=True)

def leer_gt(ruta):
    # Leemos las cajas reales de una imagen: la clase y las esquinas normalizadas (x1, y1, x2, y2)
    filas = [l.split() for l in open(ruta).read().split('\n') if l.strip()]
    if not filas:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    m = np.array(filas, dtype=float)
    xc, yc, w, h = m[:, 1], m[:, 2], m[:, 3], m[:, 4]
    return m[:, 0].astype(int), np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], axis=1)

def iou_matriz(p, g):
    # Calculamos el IoU entre cada predicción (filas) y cada caja real (columnas)
    if len(p) == 0 or len(g) == 0:
        return np.zeros((len(p), len(g)))
    x1 = np.maximum(p[:, None, 0], g[None, :, 0]); y1 = np.maximum(p[:, None, 1], g[None, :, 1])
    x2 = np.minimum(p[:, None, 2], g[None, :, 2]); y2 = np.minimum(p[:, None, 3], g[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_p = (p[:, 2] - p[:, 0]) * (p[:, 3] - p[:, 1]); area_g = (g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1])
    return inter / (area_p[:, None] + area_g[None, :] - inter + 1e-9)

def acertadas(pcls, pconf, pbox, gcls, gbox, exigir_clase):
    # Decimos, para cada caja real, si alguna detección la acertó; cada detección acierta como máximo una caja real
    hecho = np.zeros(len(gcls), dtype=bool)
    sel = pconf >= a.conf
    pcls, pconf, pbox = pcls[sel], pconf[sel], pbox[sel]
    if len(gcls) == 0 or len(pcls) == 0:
        return hecho
    iou = iou_matriz(pbox, gbox)
    if exigir_clase:
        iou = iou * (pcls[:, None] == gcls[None, :])
    for i in np.argsort(-pconf):                     # recorremos las detecciones de más a menos confianza
        libres = np.where((iou[i] >= a.iou) & ~hecho)[0]
        if len(libres):
            hecho[libres[np.argmax(iou[i, libres])]] = True
    return hecho

def proxy_oclusion(gbox):
    # Estimamos la oclusión con dos señales: los bordes de la imagen que toca la caja y cuánto la tapan otras cajas de la misma imagen
    tol = 0.005
    bordes = (gbox[:, 0] <= tol).astype(int) + (gbox[:, 1] <= tol) + (gbox[:, 2] >= 1 - tol) + (gbox[:, 3] >= 1 - tol)
    solape = np.zeros(len(gbox))
    for i in range(len(gbox)):
        area_i = (gbox[i, 2] - gbox[i, 0]) * (gbox[i, 3] - gbox[i, 1]) + 1e-9
        for j in range(len(gbox)):
            if i != j:
                ancho = max(0.0, min(gbox[i, 2], gbox[j, 2]) - max(gbox[i, 0], gbox[j, 0]))
                alto = max(0.0, min(gbox[i, 3], gbox[j, 3]) - max(gbox[i, 1], gbox[j, 1]))
                solape[i] = max(solape[i], ancho * alto / area_i)
    nivel = np.where((bordes >= 2) | (solape >= 0.5 - 1e-6), 'alto', np.where((bordes == 1) | (solape >= 0.2 - 1e-6), 'medio', 'bajo'))
    return bordes, solape, nivel

def procesar(split, model):
    # Muestreamos imágenes con animal de la partición, las predecimos por lotes y anotamos qué cajas reales se acertaron
    carpeta = f'/tmp/yolo/labels/{split}'
    con_animal = [f for f in sorted(os.listdir(carpeta)) if open(f'{carpeta}/{f}').read().strip()]
    muestra = random.Random(a.semilla).sample(con_animal, min(a.n, len(con_animal)))
    print(f"{split}: {len(con_animal)} imágenes con animal, muestreamos {len(muestra)}", flush=True)
    filas = []
    for i in range(0, len(muestra), a.lote):
        lote = muestra[i:i + a.lote]
        rutas = [f'/tmp/yolo/images/{split}/{os.path.splitext(f)[0]}.jpg' for f in lote]
        res = model.predict(rutas, conf=0.05, imgsz=640, device=a.device, verbose=False)
        for f, r in zip(lote, res):
            gcls, gbox = leer_gt(f'{carpeta}/{f}')
            pbox = r.boxes.xyxyn.cpu().numpy(); pcls = r.boxes.cls.cpu().numpy().astype(int); pconf = r.boxes.conf.cpu().numpy()
            ok_clase = acertadas(pcls, pconf, pbox, gcls, gbox, exigir_clase=True)
            ok_det = acertadas(pcls, pconf, pbox, gcls, gbox, exigir_clase=False)
            bordes, solape, nivel = proxy_oclusion(gbox)
            area = (gbox[:, 2] - gbox[:, 0]) * (gbox[:, 3] - gbox[:, 1])
            for k in range(len(gcls)):
                filas.append((split, f, int(gcls[k]), float(area[k]), int(bordes[k]), float(solape[k]), nivel[k], bool(ok_clase[k]), bool(ok_det[k])))
        del res
        if (i // a.lote) % 10 == 0:
            print(f"  {min(i + a.lote, len(muestra))}/{len(muestra)} imágenes", flush=True)
    return pd.DataFrame(filas, columns=['split', 'imagen', 'clase', 'area_rel', 'bordes', 'solape', 'oclusion', 'acierto_clase', 'acierto_deteccion'])

model = YOLO(a.weights)
df = pd.concat([procesar('cis_test', model), procesar('trans_test', model)], ignore_index=True)

# Definimos los tres grupos de tamaño con los terciles del área relativa, calculados juntando las dos particiones para que sean comparables
q1, q2 = np.quantile(df.area_rel, [1 / 3, 2 / 3])
df['tamano'] = np.where(df.area_rel < q1, 'pequena', np.where(df.area_rel < q2, 'mediana', 'grande'))
df.to_csv(f'{a.salida}/cajas.csv', index=False)

def tabla(col, orden):
    # Agrupamos por partición y por grupo, y calculamos el recall con y sin exigir la clase correcta
    t = (df.groupby(['split', col]).agg(cajas=('acierto_deteccion', 'size'), recall_clase=('acierto_clase', 'mean'),
                                         recall_deteccion=('acierto_deteccion', 'mean')).round(3).reset_index())
    t[col] = pd.Categorical(t[col], orden, ordered=True)
    return t.sort_values(['split', col]).reset_index(drop=True)

t_tam = tabla('tamano', ['pequena', 'mediana', 'grande']); t_ocl = tabla('oclusion', ['bajo', 'medio', 'alto'])
t_tam.to_csv(f'{a.salida}/tamano.csv', index=False); t_ocl.to_csv(f'{a.salida}/oclusion.csv', index=False)

print('\n===== ROBUSTEZ =====')
print(f'Umbral de confianza {a.conf}, IoU {a.iou}. Terciles del área relativa de la caja: {q1:.4f} y {q2:.4f} (fracción del área de la imagen)')
print('\nPor tamaño de caja:'); print(t_tam.to_string(index=False))
print('\nPor nivel de oclusión (proxy):'); print(t_ocl.to_string(index=False))

# Resumimos la caída entre el grupo más fácil y el más difícil, para cada partición
print('\nCaída entre grupos (recall con clase correcta):')
for split in ['cis_test', 'trans_test']:
    for col, facil, dificil, t in [('tamano', 'grande', 'pequena', t_tam), ('oclusion', 'bajo', 'alto', t_ocl)]:
        s = t[t.split == split].set_index(col).recall_clase
        f_, d_ = s.get(facil, float('nan')), s.get(dificil, float('nan'))   # si faltara un grupo, mostramos nan en vez de romper
        print(f'  {split:10s} {col:9s} {facil} {f_:.3f} -> {dificil} {d_:.3f} | caída {f_ - d_:.3f} ({100 * (f_ - d_) / max(f_, 1e-9):.1f} %)')

# Comprobaciones automáticas
print()
print('OK    ' if all(len(df[df.split == s]) > 0 for s in ['cis_test', 'trans_test']) else 'FALLO ', 'hay cajas reales en las dos particiones')
print('OK    ' if t_tam.cajas.sum() == len(df) and t_ocl.cajas.sum() == len(df) else 'FALLO ', 'los grupos suman todas las cajas')
print('OK    ' if ((t_tam.recall_clase.between(0, 1)) & (t_ocl.recall_clase.reindex(t_tam.index, fill_value=0.5).between(0, 1))).all() else 'FALLO ', 'los recalls están entre 0 y 1')
print('OK    ' if (t_tam.recall_deteccion >= t_tam.recall_clase - 0.01).all() and (t_ocl.recall_deteccion >= t_ocl.recall_clase - 0.01).all() else 'FALLO ', 'sin exigir la clase el recall nunca baja')
print('OK    ' if all(os.path.exists(f'{a.salida}/{f}') for f in ['cajas.csv', 'tamano.csv', 'oclusion.csv']) else 'FALLO ', 'resultados guardados en', a.salida)

In [ ]:
# Miramos la memoria libre antes de empezar
!free -h | head -n 2

# Analizamos la robustez por tamaño de caja y por oclusión (proxy) con el modelo final, en cámaras vistas y nuevas
!python /kaggle/working/robustez.py --weights /kaggle/working/runs/yolov8s_cct/weights/best.pt

In [ ]:
# Separamos las dos señales del proxy de oclusión (bordes que toca la caja y solape con otras cajas) para ver cuál influye más
# (solo leemos cajas.csv, no usamos la GPU)
import os
import pandas as pd
df = pd.read_csv('/kaggle/working/robustez/cajas.csv')

# Agrupamos las cajas según cuántos bordes de la imagen tocan y según cuánto las tapan otras cajas
df['grupo_bordes'] = df.bordes.clip(upper=2).map({0: '0 bordes', 1: '1 borde', 2: '2 o más bordes'})
df['grupo_solape'] = pd.cut(df.solape, [0, 1e-4, 0.2, 0.5, 1.01], right=False,
                            labels=['sin solape', 'solape < 20 %', 'solape 20-50 %', 'solape >= 50 %'])

# Calculamos el recall de cada grupo, con y sin exigir la clase correcta, y guardamos el detalle
tablas = []
for col, titulo in [('grupo_bordes', 'Por bordes de la imagen que toca la caja'), ('grupo_solape', 'Por solape con otras cajas')]:
    t = (df.groupby(['split', col], observed=True)
           .agg(cajas=('acierto_clase', 'size'), recall_clase=('acierto_clase', 'mean'), recall_deteccion=('acierto_deteccion', 'mean'))
           .round(3).reset_index().rename(columns={col: 'grupo'}))
    t.insert(0, 'senal', col)
    tablas.append(t)
    print(f'\n{titulo}:'); print(t.drop(columns='senal').to_string(index=False))
detalle = pd.concat(tablas, ignore_index=True)
detalle.to_csv('/kaggle/working/robustez/oclusion_detalle.csv', index=False)

# Comprobamos que cada señal reparte todas las cajas y que el detalle quedó guardado
for senal in ['grupo_bordes', 'grupo_solape']:
    print('OK    ' if detalle[detalle.senal == senal].cajas.sum() == len(df) else 'FALLO ', f'{senal}: los grupos suman todas las cajas ({len(df)})')
print('OK    ' if os.path.exists('/kaggle/working/robustez/oclusion_detalle.csv') else 'FALLO ', 'detalle guardado en robustez/oclusion_detalle.csv')

## 9. Evidencia sobre el fondo

Tapamos a los animales con gris y medimos si el detector sigue detectando, en cámaras vistas y nuevas.

In [ ]:
%%writefile /kaggle/working/fondo.py
# Probamos cuánto depende el detector del fondo de la escena: tapamos a los animales con gris y medimos si el modelo sigue "detectando"
import argparse, os, random
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
from ultralytics import YOLO

# Definimos los argumentos que recibe el script
ap = argparse.ArgumentParser()
ap.add_argument('--weights', required=True)
ap.add_argument('--n',       type=int,   default=1500)   # imágenes por condición y por partición
ap.add_argument('--conf',    type=float, default=0.25)   # umbral de confianza para contar una detección (el mismo del monitoreo)
ap.add_argument('--lote',    type=int,   default=50)     # imágenes por lote
ap.add_argument('--device',  default='0')
ap.add_argument('--semilla', type=int,   default=0)
ap.add_argument('--salida',  default='/kaggle/working/fondo')
a = ap.parse_args()
os.makedirs(a.salida, exist_ok=True)
GRIS = (114, 114, 114)   # el mismo gris que usa Ultralytics para rellenar los bordes

def leer_cajas(ruta):
    # Leemos las cajas reales de una imagen como esquinas normalizadas (x1, y1, x2, y2)
    filas = [l.split() for l in open(ruta).read().split('\n') if l.strip()]
    if not filas:
        return np.zeros((0, 4))
    m = np.array(filas, dtype=float)
    xc, yc, w, h = m[:, 1], m[:, 2], m[:, 3], m[:, 4]
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], axis=1)

def rellenar(img, cajas):
    # Pintamos de gris cada caja (en coordenadas normalizadas) sobre la imagen
    W, H = img.size
    d = ImageDraw.Draw(img)
    for x1, y1, x2, y2 in cajas:
        d.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], fill=GRIS)
    return img

def parches_aleatorios(tamanos, rng):
    # Creamos cajas grises del mismo tamaño que las de una imagen con animal, pero en posiciones al azar
    cajas = []
    for w, h in tamanos:
        x1, y1 = rng.uniform(0, max(1e-6, 1 - w)), rng.uniform(0, max(1e-6, 1 - h))
        cajas.append((x1, y1, x1 + w, y1 + h))
    return cajas

def detecciones(model, ruta_imgs, cajas_por_imagen):
    # Predecimos por lotes (las imágenes se modifican al vuelo para no llenar la memoria) y contamos las detecciones de cada una
    cuenta = []
    for i in range(0, len(ruta_imgs), a.lote):
        lote = []
        for ruta, cajas in zip(ruta_imgs[i:i + a.lote], cajas_por_imagen[i:i + a.lote]):
            img = Image.open(ruta).convert('RGB')
            lote.append(rellenar(img, cajas) if len(cajas) else img)
        res = model.predict(lote, conf=a.conf, imgsz=640, device=a.device, verbose=False)
        cuenta += [len(r.boxes) for r in res]
    return np.array(cuenta)

model = YOLO(a.weights)
filas = []
for split in ['cis_test', 'trans_test']:
    rng = random.Random(a.semilla)
    lab = f'/tmp/yolo/labels/{split}'
    nombres = sorted(os.listdir(lab))
    cajas = {f: leer_cajas(f'{lab}/{f}') for f in nombres}
    con_animal = [f for f in nombres if len(cajas[f])]
    vacias = [f for f in nombres if not len(cajas[f])]
    animales = rng.sample(con_animal, min(a.n, len(con_animal)))
    vac = rng.sample(vacias, min(a.n, len(vacias)))
    rutas = lambda lst: [f'/tmp/yolo/images/{split}/{os.path.splitext(f)[0]}.jpg' for f in lst]
    print(f'{split}: {len(animales)} imágenes con animal y {len(vac)} vacías', flush=True)

    # Los parches de control copian el número y el tamaño de las cajas de imágenes con animal, pero van en lugares al azar
    parches = [parches_aleatorios([(x2 - x1, y2 - y1) for x1, y1, x2, y2 in cajas[rng.choice(animales)]], rng) for _ in vac]
    sin_cambio_vac = [[] for _ in vac]
    condiciones = [
        ('A vacía', rutas(vac), sin_cambio_vac),
        ('B con animal', rutas(animales), [[] for _ in animales]),
        ('C animal tapado', rutas(animales), [cajas[f] for f in animales]),
        ('D vacía con parches', rutas(vac), parches),
    ]
    for nombre, r, c in condiciones:
        n_det = detecciones(model, r, c)
        n, k = len(n_det), int((n_det > 0).sum()); p = k / n
        filas.append(dict(split=split, condicion=nombre, imagenes=n, con_deteccion=k, pct=round(100 * p, 1),
                          ic95=round(196 * np.sqrt(p * (1 - p) / n), 1), detecciones_por_imagen=round(float(n_det.mean()), 3)))
        print(f'  {nombre:22s} {n:5d} imágenes | con detección: {100 * p:5.1f} % (±{196 * np.sqrt(p * (1 - p) / n):.1f}) | detecciones por imagen: {n_det.mean():.3f}', flush=True)

df = pd.DataFrame(filas)
df.to_csv(f'{a.salida}/resultados.csv', index=False)

print('\n===== FONDO =====')
print(f'Umbral de confianza {a.conf}. Porcentaje de imágenes con al menos una detección:')
print(df.pivot(index='condicion', columns='split', values='pct').to_string())

# Comprobaciones automáticas
print()
print('OK    ' if len(df) == 8 and (df.imagenes > 0).all() else 'FALLO ', 'cuatro condiciones en cada partición, todas con imágenes')
print('OK    ' if all(df[(df.split == s) & (df.condicion == 'A vacía')].imagenes.iloc[0] == df[(df.split == s) & (df.condicion == 'D vacía con parches')].imagenes.iloc[0] and
                      df[(df.split == s) & (df.condicion == 'B con animal')].imagenes.iloc[0] == df[(df.split == s) & (df.condicion == 'C animal tapado')].imagenes.iloc[0]
                      for s in ['cis_test', 'trans_test']) else 'FALLO ', 'las comparaciones son pareadas (mismas imágenes con y sin cambio)')
print('OK    ' if df.pct.between(0, 100).all() else 'FALLO ', 'los porcentajes están entre 0 y 100')
print('OK    ' if all(df[(df.split == s) & (df.condicion == 'B con animal')].pct.iloc[0] > df[(df.split == s) & (df.condicion == 'C animal tapado')].pct.iloc[0]
                      for s in ['cis_test', 'trans_test']) else 'REVISAR', 'tapar al animal reduce las detecciones (lo esperado)')
print('OK    ' if os.path.exists(f'{a.salida}/resultados.csv') else 'FALLO ', 'resultados guardados en', a.salida)

In [ ]:
# Miramos la memoria libre antes de empezar
!free -h | head -n 2

# Medimos cuánto depende el modelo del fondo: imágenes vacías, con animal, con el animal tapado y vacías con parches de control
!python /kaggle/working/fondo.py --weights /kaggle/working/runs/yolov8s_cct/weights/best.pt

## 10. Guardamos los resultados

Empaquetamos todo en dos zips: uno completo, con los pesos, y uno ligero, sin ellos.

In [ ]:
# Juntamos pesos, métricas, reportes y scripts en un solo zip y comprobamos que lo esencial esté dentro
import os, glob, zipfile
R = '/kaggle/working/runs/yolov8s_cct'
rutas = (glob.glob(f'{R}/weights/*.pt') + glob.glob(f'{R}/*.csv') + glob.glob(f'{R}/*.png') + glob.glob(f'{R}/*.jpg') +
         glob.glob('/kaggle/working/eval/*.csv') + glob.glob('/kaggle/working/monitoreo/*.csv') + glob.glob('/kaggle/working/preds/*.csv') +
         glob.glob('/kaggle/working/robustez/*.csv') + glob.glob('/kaggle/working/fondo/*.csv') +
         glob.glob('/kaggle/working/*.py') + glob.glob('/kaggle/working/*.log') +
         ['/kaggle/working/meta_images.csv', '/kaggle/working/data.yaml'])
zip_ruta = '/kaggle/working/resultados_entrenamiento.zip'
with zipfile.ZipFile(zip_ruta, 'w', zipfile.ZIP_DEFLATED) as z:
    for r in rutas:
        if os.path.exists(r):
            z.write(r, os.path.relpath(r, '/kaggle/working'))
    nombres = z.namelist()

# Comprobamos que el zip traiga lo esencial, incluidos el monitoreo, la robustez y el fondo
for esencial in ['runs/yolov8s_cct/weights/best.pt', 'runs/yolov8s_cct/results.csv', 'eval/globales.csv',
                 'eval/brecha_por_clase.csv', 'monitoreo/resumen_cam46.csv', 'monitoreo/comparacion_cam46.csv',
                 'robustez/tamano.csv', 'robustez/oclusion.csv', 'robustez/oclusion_detalle.csv', 'fondo/resultados.csv', 'meta_images.csv']:
    print('OK    ' if esencial in nombres else 'FALLO ', esencial)
print(f"\nZIP: {os.path.getsize(zip_ruta)/1e6:.1f} MB, {len(nombres)} archivos")

In [ ]:
# Armamos un zip ligero, sin los pesos, con lo necesario para escribir el informe y montar el repositorio
import os, glob, zipfile
R = '/kaggle/working/runs/yolov8s_cct'
rutas = (glob.glob(f'{R}/*.csv') + glob.glob(f'{R}/*.png') + glob.glob(f'{R}/*.jpg') + glob.glob(f'{R}/args.yaml') +
         glob.glob('/kaggle/working/eval/*.csv') + glob.glob('/kaggle/working/monitoreo/*.csv') + glob.glob('/kaggle/working/preds/*.csv') +
         glob.glob('/kaggle/working/robustez/*.csv') + glob.glob('/kaggle/working/fondo/*.csv') + glob.glob('/kaggle/working/*.py') +
         ['/kaggle/working/mem.log', '/kaggle/working/data.yaml'])
zip_ligero = '/kaggle/working/resultados_ligeros.zip'
with zipfile.ZipFile(zip_ligero, 'w', zipfile.ZIP_DEFLATED) as z:
    for r in rutas:
        if os.path.exists(r):
            z.write(r, os.path.relpath(r, '/kaggle/working'))
    nombres = z.namelist()

# Comprobamos que traiga las métricas, los reportes y los ocho scripts
esenciales = ['runs/yolov8s_cct/results.csv', 'runs/yolov8s_cct/args.yaml', 'eval/globales.csv', 'eval/por_clase.csv', 'eval/brecha_por_clase.csv',
              'monitoreo/comparacion_cam46.csv', 'robustez/tamano.csv', 'robustez/oclusion_detalle.csv', 'robustez/cajas.csv', 'fondo/resultados.csv',
              'prep_data.py', 'train.py', 'verificar_datos.py', 'eval.py', 'monitor.py', 'predict.py', 'robustez.py', 'fondo.py']
for e in esenciales:
    print('OK    ' if e in nombres else 'FALLO ', e)

# Miramos cuánto pesa, para saber si cabe en el chat
tam = os.path.getsize(zip_ligero) / 1e6
print(f"\nZIP LIGERO: {tam:.1f} MB, {len(nombres)} archivos")
print('OK    ' if tam < 25 else 'AVISO ', 'tamaño razonable para subirlo al chat (menos de 25 MB)')
print('OK    ' if not any(n.endswith('.pt') for n in nombres) else 'FALLO ', 'el zip no incluye los pesos')